<a href="https://colab.research.google.com/github/jagadeesh-usd/composer-classification/blob/jag-dev/notebooks/01_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Purpose: Load MIDI files, process them into sequences, and save for training.

In [11]:
# 1. Import Required Libraries
import os
import time
import glob
import pickle
from multiprocessing import Pool, cpu_count
from collections import Counter

from music21 import converter, instrument, note, chord

import pandas as pd
from sklearn.model_selection import train_test_split

In [12]:
# 2. Configuration
DATA_PATH = "/content/drive/My Drive/composer_project/data/midi_files/train"
COMPOSERS = ["Bach", "byrd", "Chopin", "Handel", "Mozart", "schumann"]
SEQUENCE_LENGTH = 100 # The length of a single training sequence
VOCAB_SIZE = 2000 # The number of unique notes/chords to keep

In [13]:
# 3. Data Processing Functions

def get_midi_files(data_path, composers):
    """Finds all MIDI files and creates a DataFrame of file paths and labels."""
    all_files = []
    for composer in composers:
        composer_path = os.path.join(data_path, composer.lower())
        if os.path.isdir(composer_path):
            search_pattern = os.path.join(composer_path, '*.mid*')
            files = glob.glob(search_pattern)
            for file in files:
                all_files.append({'path': file, 'composer': composer})
            print(f"Found {len(files)} files for {composer}")
        else:
            print(f"Warning: Directory not found for {composer}: {composer_path}")
    return pd.DataFrame(all_files)

def extract_notes(file_path):
    """Extracts notes and chords from a single MIDI file."""
    notes = []
    try:
        midi = converter.parse(file_path)
        parts = instrument.partitionByInstrument(midi)
        notes_to_parse = parts.parts[0].recurse() if parts else midi.flat.notes
        for element in notes_to_parse:
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                notes.append('.'.join(str(n) for n in element.normalOrder))
    except Exception as e:
        print(f"  Error processing {os.path.basename(file_path)}: {e}")
    return notes

In [14]:
# 4. Main Processing Logic

# Create a DataFrame of all file paths and their composers
files_df = get_midi_files(DATA_PATH, COMPOSERS)

# Split the list of FILES into training (80%) and testing (20%) sets
train_files_df, test_files_df = train_test_split(
    files_df,
    test_size=0.2,
    random_state=42,
    stratify=files_df['composer'] # Ensure composer balance in splits
)

print(f"\nSplitting by file: {len(train_files_df)} train, {len(test_files_df)} test files.")

def create_sequences_from_files(df):
    """Process a dataframe of files to create note sequences."""
    all_notes = []
    sequences = []
    labels = []
    for index, row in df.iterrows():
        notes = extract_notes(row['path'])
        if len(notes) > SEQUENCE_LENGTH:
            all_notes.extend(notes)
            for i in range(0, len(notes) - SEQUENCE_LENGTH, 1):
                sequences.append(notes[i: i + SEQUENCE_LENGTH])
                labels.append(row['composer'])
    return sequences, labels, all_notes

print("\nProcessing training files to create sequences...")
train_sequences, train_labels, train_all_notes = create_sequences_from_files(train_files_df)

print("\nProcessing testing files to create sequences...")
test_sequences, test_labels, _ = create_sequences_from_files(test_files_df)

print(f"\nTraining sequences: {len(train_sequences)}")
print(f"Testing sequences: {len(test_sequences)}")


Found 42 files for Bach
Found 42 files for byrd
Found 41 files for Chopin
Found 41 files for Handel
Found 41 files for Mozart
Found 38 files for schumann

Splitting by file: 196 train, 49 test files.

Processing training files to create sequences...


/usr/local/lib/python3.11/dist-packages/music21/midi/translate.py:874: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=19, channel=None, data=b'Sequenced by Ken Whitcomb \xa91998'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/music21/midi/translate.py:874: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=13, channel=None, data=b'Sequenced by Ken Whitcomb \xa91997'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/music21/midi/translate.py:874: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, channel=None, data=b'SCHUMANN  FANTASIE  C Dur  Op.17  1st mov.       \x89\x89\x91t\x81@\x8f\xac\x8cI\x8d\x8e\x97T'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/music21/midi/translate.py:874: TranslateWarning: Unab


Processing testing files to create sequences...


/usr/local/lib/python3.11/dist-packages/music21/midi/translate.py:874: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=4, channel=None, data=b'Georg Friedrich H\xe4ndel'>; getting generic Instrument
  warnings.warn(



Training sequences: 210101
Testing sequences: 49105


In [15]:
# 5. Create Vocabulary
print("\nCreating vocabulary from training data only...")
note_counts = Counter(train_all_notes)
most_common_notes = [n for n, c in note_counts.most_common(VOCAB_SIZE - 1)]
most_common_notes.append('UNK')

note_to_int = {note: i for i, note in enumerate(most_common_notes)}
int_to_note = {i: note for note, i in note_to_int.items()}
print(f"Vocabulary size: {len(note_to_int)}")


Creating vocabulary from training data only...
Vocabulary size: 873


In [16]:
# 6. Save Processed Data
print("\nSaving processed data...")
SAVE_DIR = "/content/drive/My Drive/composer_project/processed_data_split"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save training data
with open(os.path.join(SAVE_DIR, 'train_sequences.pkl'), 'wb') as f:
    pickle.dump(train_sequences, f)
with open(os.path.join(SAVE_DIR, 'train_labels.pkl'), 'wb') as f:
    pickle.dump(train_labels, f)

# Save testing data
with open(os.path.join(SAVE_DIR, 'test_sequences.pkl'), 'wb') as f:
    pickle.dump(test_sequences, f)
with open(os.path.join(SAVE_DIR, 'test_labels.pkl'), 'wb') as f:
    pickle.dump(test_labels, f)

# Save preprocessing info
preprocessing_data = {
    'note_to_int': note_to_int,
    'int_to_note': int_to_note,
    'composers': COMPOSERS,
    'sequence_length': SEQUENCE_LENGTH,
    'vocab_size': VOCAB_SIZE
}
with open(os.path.join(SAVE_DIR, 'preprocessing_data.pkl'), 'wb') as f:
    pickle.dump(preprocessing_data, f)

print(f"Data saved to {SAVE_DIR}. Preprocessing complete.")


Saving processed data...
Data saved to /content/drive/My Drive/composer_project/processed_data_split. Preprocessing complete.
